In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/churn-siniflandirma/train.csv
/kaggle/input/competitions/churn-siniflandirma/test.csv


In [5]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# 1. Verileri Okuma
train = pd.read_csv('/kaggle/input/competitions/churn-siniflandirma/train.csv')
test = pd.read_csv('/kaggle/input/competitions/churn-siniflandirma/test.csv')

id_col = 'customerid' 
target_col = 'churn'

X = train.drop(columns=[id_col, target_col], errors='ignore')
y = train[target_col]
X_test = test.drop(columns=[id_col], errors='ignore')

# 2. Kategorik Dönüşüm
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 3. Cross-Validation Ayarları (5 Katmanlı)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Skorları ve tahminleri saklamak için listeler oluşturuyoruz
oof_preds = np.zeros(len(train)) # Out-of-fold tahminler
test_preds = np.zeros(len(test))  # Test verisi tahminleri
fold_scores = []

print("=== 5-Fold Cross Validation Başlıyor ===\n")

# 4. Döngü ile her katmanı eğitiyoruz
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Her katman için taze bir model tanımlıyoruz
    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        random_state=42 + fold, # Her foldda küçük bir çeşitlilik
        n_jobs=-1,
        verbose=-1
    )
    
    # Modeli bu katman için eğitiyoruz
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[]
    )
    
    # Doğrulama tahminlerini al ve kaydet
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    # Bu katmanın skorunu hesapla
    fold_score = roc_auc_score(y_val, val_preds)
    fold_scores.append(fold_score)
    print(f"Fold {fold+1} ROC AUC Skoru: {fold_score:.5f}")
    
    # Test verisi için tahmin yap ve 5'te 1'ini ekle (Ortalama almak için)
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

# 5. Genel Sonuçları Yazdıralım
print("\n=============================================")
print(f"5 FOLD ORTALAMA ROC AUC SKORUN: {np.mean(fold_scores):.5f}")
print("=============================================\n")

# 6. Güvenli Gönderim Dosyasını Hazırlama
submission = pd.DataFrame({
    id_col: test[id_col],
    target_col: test_preds # 5 modelin ortalaması alınmış tahminler
})

submission.to_csv('submission.csv', index=False)
print("Cross-Validation destekli submission.csv başarıyla oluşturuldu!")

=== 5-Fold Cross Validation Başlıyor ===

Fold 1 ROC AUC Skoru: 0.82607
Fold 2 ROC AUC Skoru: 0.81532
Fold 3 ROC AUC Skoru: 0.82677
Fold 4 ROC AUC Skoru: 0.83881
Fold 5 ROC AUC Skoru: 0.83343

5 FOLD ORTALAMA ROC AUC SKORUN: 0.82808

Cross-Validation destekli submission.csv başarıyla oluşturuldu!
